In [1]:
from pyspark.sql import SparkSession
import os
# 1. Direct Spark to find winutils.exe inside your existing manual Hadoop 3 environment layout
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["spark.home.dir"] = "C:\\Spark\\spark-4.0.1-bin-hadoop3"
spark = (SparkSession.builder
        .appName("Practical_Test")
        .master("local[4]")
        .config("spark.executor.memory", "8g")
        .config("spark.driver.memory", "4g")
        .config("spark.python.worker.faulthandler.enabled", "true")
        .config("spark.jars.packages", "io.delta:delta-spark_2.13:4.0.1")
        .config("spark.sql.execution.pyspark.udf.faulthandler.enabled", "true")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.sql.debug.maxToStringFields", 2000).getOrCreate())

In [ ]:
# PART B — Spark config for this exercise (performance scenario)
spark.conf.set("spark.sql.adaptive.enabled", "false")
print(spark.conf.get("spark.sql.shuffle.partitions"))

In [ ]:
#     event_id        STRING
#     op              STRING    r = initial snapshot row, c = create, u = update, d = delete
#     source_seq      BIGINT    commit order in the source system (higher = committed later)
#     commit_ts       STRING    commit time, ISO-8601 UTC with milliseconds, e.g. 2026-09-06T03:50:35.000Z
#     customer_id     STRING
#     effective_from  STRING    business date (not used in this exercise)
#     changed_fields  ARRAY<STRING>
#     before, after   STRUCT<segment STRING, account_manager STRING, city STRING, credit_limit INT, status STRING>
#                     (before is null for r/c; after is null for d)
from pyspark.sql.types import StringType, IntegerType, StructType, StructField, ArrayType


before_after_schema = StructType([StructField("segment", StringType()),
                                  StructField("account_manager", StringType()),
                                  StructField("city", StringType()),
                                  StructField("credit_limit", IntegerType()),
                                  StructField("status", StringType())])
crm_schema = StructType([
    StructField("event_id", StringType()),
    StructField("op", StringType()),
    StructField("source_seq", IntegerType()),
    StructField("commit_ts", StringType()),
    StructField("customer_id", StringType()),
    StructField("effective_from", StringType()),
    StructField("changed_fields", ArrayType(StringType())),
    StructField("before", before_after_schema),
    StructField("after", before_after_schema),
    StructField("_corrupt_record", StringType())
])


In [ ]:
from pyspark.sql.functions import col
crm_df  = (spark.read
           .format("json")
           .schema(crm_schema)
           .option("columnNameOfCorruptRecord","_corrupt_record")
           .load("file:///D:/Project_AI/PySpark_Hands_On/Input/landing/crm/customer_changes/*/*.jsonl"))

crm_df.cache()
total_json_count = crm_df.count()
crm_df = crm_df.withColumn("source_file_name", col("_metadata.file_name"))
file_read_count = crm_df.select("source_file_name").distinct().count()
invalid_json_count = crm_df.filter(col("_corrupt_record").isNotNull()).count()
crm_df = crm_df.filter(col("_corrupt_record").isNull())
crm_df = crm_df.dropDuplicates(["event_id"])
unique_event_id_count = crm_df.count()
print(f"files read: {file_read_count}")
print(f"input lines: {total_json_count}")
print(f"ignored lines:  {invalid_json_count}")
print(f"distinct events: {unique_event_id_count}")
print(f"duplicate deliveries removed {total_json_count - invalid_json_count - unique_event_id_count}")

In [ ]:
from pyspark.sql.functions import size
no_change_count = crm_df.filter((col('op')=='u') & (size(col("changed_fields")) == 0)).count()
crm_df = crm_df.filter((col('op')!='u') | (size(col("changed_fields")) != 0))